In [1]:
import torch
import torch.nn as nn
import numpy as np
import os
import sys
sys.dont_write_bytecode = True
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../bin')))
from model import Surfeleton, TransformerModelold,TransformerModel
from atomsurf.utils.wrappers import get_default_model
from atomsurf.utils.data_utils import AtomBatch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [2]:
args = {
    'embed_dim': 128,         
    'num_heads': 4,           
    'ffn_dim': 256,           
    'num_layers': 4,          
    'dropout': 0.1,            
    'surface_dim': 52,        # 
    'graph_dim': 512,        # 
    'n_block': 4,            # 
    'model_dim': 128        # 
}
transformer_model = TransformerModel(
    embed_dim=args['embed_dim'],
    num_heads=args['num_heads'],
    ffn_dim=args['ffn_dim'],
    num_layers=args['num_layers'],
    dropout=args['dropout']
)

atomsurf_model = get_default_model(
    args['surface_dim'],
    args['graph_dim'],
    model_dim=args['model_dim'], 
    dropout=args['dropout'], 
    n_block=args['n_block']
)

model = Surfeleton(atomsurf_model, transformer_model).to(device)

In [ ]:
model_path = "../bin/model_bin/CATH_4_2_final.pth"
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("successfully load model")

successfully load model


: 

In [10]:
pt_path = "../example/if1_pt_dataset/1r26_A.pt"
protein,native_seq = torch.load(pt_path)["pdata"],torch.load(pt_path)["seq"]
res = model(AtomBatch.from_data_list([protein]).to(device),device)


In [11]:
def onehot_to_seq_reduction(onehot_matrix):

    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    aa_dict = {i: amino_acids[i] for i in range(len(amino_acids))}
    

    seq = [aa_dict[np.argmax(onehot)] for onehot in onehot_matrix]
    
    return ''.join(seq)
    
def onehot_to_seq_gen(onehot_matrix, temperature=1.0):

    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    epsilon = 1e-8  

    safe_temp = max(temperature, 1e-3)
    logits = onehot_matrix / safe_temp

    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    probs = exp_logits / (np.sum(exp_logits, axis=1, keepdims=True) + epsilon)

    if np.any(np.isnan(probs)) or np.any(np.isinf(probs)):
        raise ValueError("Numerical instability: probabilities contain NaN or inf.")

    sampled_indices = [np.random.choice(len(amino_acids), p=probs[i]) for i in range(len(probs))]
    seq = ''.join([amino_acids[idx] for idx in sampled_indices])
    
    return seq

In [12]:
new_seq = onehot_to_seq_reduction(res[0].detach().cpu().numpy())
gen_seq = onehot_to_seq_gen(res[0].detach().cpu().numpy(), temperature=0.5)


In [13]:
print(gen_seq)
print(new_seq)
print(native_seq)



IKKRARYPSPVEIYSDEQCRNIVSQDILTVANFTAVYCGTCKTIERPLEKLCREYPTVEFLFVDADNNSEIVAKERVLELPTFIIMQSGKFLGKVVGANPGQLRQKLRNIIKD
IKKRARYPSPVEIYSDEQCRDIVSQDILTVADFTAVYCGTCKTIERPMEKLCREFPTVEFLRVDADNNSEIVAKERVLQLPTFVIMMSGKFLGKVVGANPGQLREKLRNIIKD
IRMRARYPSVVDVYSVEQFRNIMSEDILTVAWFTAVWCGPCKTIERPMEKIAYEFPTVKFAKVDADNNSEIVSKCRVLQLPTFIIARSGKMLGHVIGANPGMLRQKLRDIIKD
